### Task 1: Download and Load the Dataset

**Question:**
> *How do we programmatically download, extract, and load the UCI Sentiment Labeled Sentences dataset into a Pandas DataFrame for NLP analysis using Python?*

**Explanation:**
* **Fetching Data (`urllib.request`)**: The script checks if the UCI sentiment dataset archive is present locally and downloads it directly from the UCI Machine Learning Repository if missing[cite: 1].
* **Extraction (`zipfile`)**: It unpacks the downloaded ZIP archive to access the underlying text files[cite: 1].
* **Loading into Pandas (`pd.read_csv`)**: The review file (`yelp_labelled.txt`) is formatted as tab-separated values, where each row contains a text review followed by a binary sentiment label (`1` for positive, `0` for negative). We load this into a structured Pandas DataFrame using `sep='\t'`[cite: 1].

In [16]:
import os
import urllib.request
import zipfile
import pandas as pd

# 1. Define dataset URL and paths
DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
ZIP_FILE = "sentiment_labelled_sentences.zip"
EXTRACTED_DIR = "sentiment_labelled_sentences"

def download_and_load_data():
    # Download zip file if it doesn't already exist
    if not os.path.exists(ZIP_FILE):
        print("Downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_FILE)
        print("Download complete.")

    # Extract zip file
    if not os.path.exists(EXTRACTED_DIR):
        print("Extracting dataset...")
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(EXTRACTED_DIR)
        print("Extraction complete.")

    # Path to the Yelp reviews file included in the dataset
    yelp_file_path = os.path.join(EXTRACTED_DIR, "sentiment labelled sentences", "yelp_labelled.txt")

    # Load into a Pandas DataFrame (tab-separated values, no header)
    print(f"Loading dataset from: {yelp_file_path}")
    df = pd.read_csv(yelp_file_path, sep='\t', header=None, names=['review', 'sentiment'])

    return df

# Execute the function and display the output
df = download_and_load_data()
print("\nDataset Shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
display(df.head())

Loading dataset from: sentiment_labelled_sentences\sentiment labelled sentences\yelp_labelled.txt

Dataset Shape: (1000, 2)

First 5 rows of the dataset:


,review,sentiment
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


### Task 2: Perform Basic Text Preprocessing

**Question:**
> *How do we clean raw natural language text by converting it to lowercase, removing punctuation/special characters, eliminating extra whitespaces, and creating a new clean-text column for comparison?*

**Explanation:**
* **Lowercasing**: Uniformly converts all characters to lowercase so that words like "Great" and "great" are treated identically by the model[cite: 18].
* **Punctuation & Special Character Removal (`re`)**: Uses Regular Expressions to strip out symbols, numbers, and punctuation marks that add noise rather than semantic meaning[cite: 18].
* **Whitespace Normalization**: Removes trailing, leading, or multiple consecutive spaces to ensure clean token boundaries.
* **Comparison**: Creates a new column (`clean_review`) alongside the original text to verify that the cleaning steps preserved the core semantic meaning while removing unwanted noise[cite: 1].

In [17]:
import re

def preprocess_text(text):
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters using regex (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply preprocessing to create a new 'clean_review' column
df['clean_review'] = df['review'].apply(preprocess_text)

# Compare original vs preprocessed text side by side
print("Comparison of Original vs Preprocessed Text:\n")
comparison_df = df[['review', 'clean_review']].head(10)
display(comparison_df)

Comparison of Original vs Preprocessed Text:



,review,clean_review
0,Wow... Loved this place.,wow loved this place
1,Crust is not good.,crust is not good
2,Not tasty and the texture was just nasty.,not tasty and the texture was just nasty
3,Stopped by during the late May bank holiday of...,stopped by during the late may bank holiday of...
4,The selection on the menu was great and so wer...,the selection on the menu was great and so wer...
5,Now I am getting angry and I want my damn pho.,now i am getting angry and i want my damn pho
6,Honeslty it didn't taste THAT fresh.),honeslty it didnt taste that fresh
7,The potatoes were like rubber and you could te...,the potatoes were like rubber and you could te...
8,The fries were great too.,the fries were great too
9,A great touch.,a great touch


### Task 3: Explore the Text Data

**Question:**
> *How can we explore our dataset to display positive and negative examples, analyze class distribution, and identify frequently occurring words in our text corpus?*

**Explanation:**
* **Class Distribution Check**: Verifies the balance between positive (`1`) and negative (`0`) sentiment labels in the dataset.
* **Positive/Negative Examples**: Inspects sample rows from each category to understand the language used in user reviews.
* **Word Frequency Analysis (`Counter`)**: Aggregates all words across the corpus to find the most common terms, which helps identify dominant themes before vectorization.

In [9]:
from collections import Counter

# 1. Check sentiment class distribution
print("Sentiment Class Distribution:")
print(df['sentiment'].value_counts())

# 2. Display positive and negative examples
print("\n--- Sample Positive Review ---")
print(df[df['sentiment'] == 1]['review'].iloc[0])

print("\n--- Sample Negative Review ---")
print(df[df['sentiment'] == 0]['review'].iloc[0])

# 3. Identify frequently occurring words in cleaned reviews
all_words = " ".join(df['clean_review']).split()
word_counts = Counter(all_words)

print("\nTop 10 Most Common Words:")
for word, count in word_counts.most_common(10):
    print(f"{word}: {count}")

Sentiment Class Distribution:
sentiment
1    500
0    500
Name: count, dtype: int64

--- Sample Positive Review ---
Wow... Loved this place.

--- Sample Negative Review ---
Crust is not good.

Top 10 Most Common Words:
the: 584
and: 389
i: 304
was: 295
a: 237
to: 219
is: 171
this: 143
it: 132
of: 127


### Task 4: Split the Dataset

**Question:**
> *Why and how do we split our dataset into training and testing sets for machine learning model development?*

**Explanation:**
* **Train/Test Split (`train_test_split`)**: We divide our data so the machine learning model learns patterns from the training set while holding out the testing set to evaluate its generalization performance on unseen data.
* **Stratification (`stratify=y`)**: Ensures that both the training and testing subsets maintain the same proportion of positive and negative sentiment labels as the original dataset.

In [10]:
from sklearn.model_selection import train_test_split

# Define features (X) and labels (y)
X = df['clean_review']
y = df['sentiment']

# Split into 80% training and 20% testing sets, maintaining class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")
print("\nTraining label distribution:\n", y_train.value_counts(normalize=True))

Training set size: 800 samples
Testing set size: 200 samples

Training label distribution:
 sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


### Task 6: Train a Machine Learning Model

**Question:**
> *How do we train a traditional machine learning classifier (such as Logistic Regression) using our Bag of Words vector representation for sentiment classification[cite: 1]?*

**Explanation:**
* **Classifier Choice (`LogisticRegression`)**: Logistic regression is a robust, interpretable baseline model for binary text classification tasks.
* **Model Training (`fit`)**: The model learns the mapping between word counts (features) and sentiment labels (positive or negative) using the training dataset[cite: 1].

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Split the dataset (Task 4)
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Create BOW representation (Task 5)
vectorizer = CountVectorizer(max_features=2500)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# 3. Train the ML model (Task 6)
model = LogisticRegression(random_state=42)
model.fit(X_train_bow, y_train)

print("Data split, vectorization, and model training completed successfully!")
print(f"Training feature matrix shape: {X_train_bow.shape}")

Data split, vectorization, and model training completed successfully!
Training feature matrix shape: (800, 1783)


### Task 7: Evaluate Model Performance

**Question:**
> *How do we evaluate our trained sentiment classification model using metrics such as Accuracy, Precision, Recall, and F1-Score on the test set[cite: 1]?*

**Explanation:**
* **Accuracy**: Measures the overall percentage of correct predictions out of all test samples.
* **Precision**: Measures the proportion of positive predictions that were actually correct (minimizes false positives).
* **Recall**: Measures the proportion of actual positive samples that were correctly identified (minimizes false negatives).
* **F1-Score**: The harmonic mean of precision and recall, providing a balanced metric especially useful for classification tasks.

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Predict labels for the test set
y_pred = model.predict(X_test_bow)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("--- Model Evaluation Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

--- Model Evaluation Metrics ---
Accuracy:  0.8500
Precision: 0.8365
Recall:    0.8700
F1-Score:  0.8529

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.85       100
           1       0.84      0.87      0.85       100

    accuracy                           0.85       200
   macro avg       0.85      0.85      0.85       200
weighted avg       0.85      0.85      0.85       200



### Task 8: Test the Model with a New Sentence

**Question:**
> *How do we preprocess and vectorize a completely new, unseen sentence to predict its sentiment using our trained machine learning model[cite: 1]?*

**Explanation:**
* **Pipeline Consistency**: Any new input sentence must go through the exact same preprocessing steps (lowercasing, punctuation removal) and vectorization (`transform`) that were used on the training data.
* **Inference (`predict`)**: The vector is passed to our trained model to output a sentiment classification prediction (`1` for positive, `0` for negative).

In [20]:
def predict_sentiment(sentence):
    # 1. Preprocess the new sentence using our existing function
    clean_sentence = preprocess_text(sentence)

    # 2. Vectorize using the fitted vectorizer
    sentence_bow = vectorizer.transform([clean_sentence])

    # 3. Predict sentiment
    prediction = model.predict(sentence_bow)[0]

    sentiment_label = "Positive 😊" if prediction == 1 else "Negative 😞"
    return sentiment_label

# Test with new sentences
test_sentences = [
    "The food was absolutely wonderful and the service was top notch!",
    "Terrible experience, I will never come back here again."
]

print("--- Testing Model with New Sentences ---")
for sentence in test_sentences:
    result = predict_sentiment(sentence)
    print(f"\nReview: '{sentence}'")
    print(f"Predicted Sentiment: {result}")

--- Testing Model with New Sentences ---

Review: 'The food was absolutely wonderful and the service was top notch!'
Predicted Sentiment: Positive 😊

Review: 'Terrible experience, I will never come back here again.'
Predicted Sentiment: Negative 😞
